# Healthcare Claims — SQL Analytics Layer

**Layer:** SQL Analytics (Business Reporting)  
**Input:** Curated claims data from the ETL pipeline (notebook 01)  
**Purpose:** Answer business questions that support program management, cost control, and utilization monitoring.

### Query Groups
1. **Cost Analysis** — spend KPIs, trends, plan comparisons
2. **Utilization Analysis** — behavioral health, provider performance
3. **Risk / High-Cost Patients** — member stratification, readmission signals
4. **Data Quality** — anomaly detection queries

### Compatibility
All queries run on **SQLite** here for local demo. The same SQL is compatible with **Databricks SQL, Snowflake, and BigQuery** with minor dialect adjustments (e.g. `STRFTIME` → `DATE_TRUNC`).

In [ ]:
import sqlite3
import pandas as pd

pd.set_option('display.float_format', '{:,.2f}'.format)

# Load curated data into in-memory SQLite
# In production: connect to Databricks SQL / Snowflake instead
conn = sqlite3.connect(':memory:')
df   = pd.read_csv('../data/sample_claims.csv')
df.to_sql('claims', conn, index=False, if_exists='replace')

def q(sql, title=''):
    if title:
        print(f'\n{title}')
    return pd.read_sql(sql, conn)

print(f'Loaded {len(df)} claims into analytics layer')

---
## Section 1: Cost Analysis

These queries answer the most common questions from program managers and finance teams: how much are we spending, where is it going, and how does it compare across plans?

In [ ]:
# Business question: What are our overall spend and denial KPIs?
# Why it matters: Baseline metrics every program report starts with.
# Denial rate > 10% typically warrants investigation.

q("""
SELECT
    COUNT(*)                                                          AS total_claims,
    COUNT(CASE WHEN claim_status='PAID'   THEN 1 END)                 AS paid_claims,
    COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)                 AS denied_claims,
    ROUND(100.0 * COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)
                / COUNT(*), 2)                                        AS denial_rate_pct,
    ROUND(SUM(billed_amount), 2)                                      AS total_billed,
    ROUND(SUM(paid_amount), 2)                                        AS total_paid,
    ROUND(100.0 * SUM(paid_amount) / SUM(billed_amount), 2)           AS avg_payment_rate_pct
FROM claims
""", 'Q1 — Overall KPIs')

In [ ]:
# Business question: How is spend trending month over month?
# Why it matters: Sudden spikes in inpatient (IP) spend often indicate
# seasonal illness, coding changes, or a high-cost member event.
# Grouping by claim_type separates cost drivers.

q("""
SELECT
    STRFTIME('%Y-%m', service_date)   AS service_month,
    claim_type,
    COUNT(*)                          AS claim_count,
    ROUND(SUM(paid_amount), 2)        AS total_paid,
    ROUND(AVG(paid_amount), 2)        AS avg_paid_per_claim
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
ORDER BY 1, 2
""", 'Q2 — Monthly Spend Trend by Claim Type')

In [ ]:
# Business question: How does spend and efficiency compare across plans?
# Why it matters: PMPM (per member per month) is the standard managed care
# metric for plan performance. Large variation signals utilization differences
# or enrollment mix issues that warrant further investigation.

q("""
SELECT
    state_code,
    plan_id,
    COUNT(DISTINCT member_id)                                          AS unique_members,
    COUNT(*)                                                           AS total_claims,
    ROUND(SUM(paid_amount), 2)                                         AS total_paid,
    ROUND(SUM(paid_amount) / COUNT(DISTINCT member_id), 2)             AS pmpm_proxy
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
ORDER BY total_paid DESC
""", 'Q3 — Spend by State and Plan (PMPM)')

---
## Section 2: Utilization Analysis

Utilization queries track *how* the system is being used — which provider types are driving volume, where denials are concentrated, and how behavioral health (a federally mandated parity category) compares.

In [ ]:
# Business question: Which provider types have the highest denial rates?
# Why it matters: High denial rates for a specific provider type usually
# indicate a billing or coding issue — a starting point for
# provider education or targeted audit.

q("""
SELECT
    provider_type,
    COUNT(*)                                                             AS total_claims,
    COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)                    AS denied,
    ROUND(100.0 * COUNT(CASE WHEN claim_status='DENIED' THEN 1 END)
                / COUNT(*), 2)                                           AS denial_rate_pct,
    ROUND(SUM(billed_amount) - SUM(paid_amount), 2)                      AS total_underpaid
FROM claims
GROUP BY 1
ORDER BY denial_rate_pct DESC
""", 'Q4 — Denial Rate by Provider Type')

In [ ]:
# Business question: What is behavioral health spend by state?
# Why it matters: BH services are subject to federal mental health parity
# requirements. Tracking BH spend separately ensures compliance reporting
# and identifies states where utilization may be under- or over-reported.

q("""
SELECT
    state_code,
    COUNT(*)                          AS bh_claims,
    COUNT(DISTINCT member_id)         AS unique_members,
    ROUND(SUM(paid_amount), 2)        AS total_paid,
    ROUND(AVG(paid_amount), 2)        AS avg_paid
FROM claims
WHERE provider_type = 'BEHAVIORAL'
  AND claim_status  = 'PAID'
GROUP BY 1
ORDER BY total_paid DESC
""", 'Q5 — Behavioral Health Utilization by State')

---
## Section 3: Risk / High-Cost Patient Identification

A small percentage of members typically drives a disproportionate share of spend. These queries surface them for care management outreach — a core function of Medicaid managed care programs.

In [ ]:
# Business question: Who are our highest-cost members?
# Why it matters: In most Medicaid programs, the top 5% of members
# account for ~50% of spend. Identifying them enables targeted
# care management interventions that reduce cost and improve outcomes.
# Inpatient visit count is included as a risk signal.

q("""
SELECT
    member_id,
    state_code,
    COUNT(*)                                           AS total_claims,
    COUNT(CASE WHEN claim_type='IP' THEN 1 END)        AS inpatient_visits,
    ROUND(SUM(paid_amount), 2)                         AS total_paid,
    ROUND(AVG(paid_amount), 2)                         AS avg_per_claim
FROM claims
WHERE claim_status = 'PAID'
GROUP BY 1, 2
HAVING SUM(paid_amount) > 5000
ORDER BY total_paid DESC
""", 'Q6 — High-Cost Member Identification')

In [ ]:
# Business question: Which members may have had a preventable readmission?
# Why it matters: 30-day readmissions are a key quality metric tied to
# CMS value-based payment programs. Early identification supports
# transition-of-care outreach.
#
# Note: This is a proxy — a true readmission measure requires
# discharge dates. This flags members with two IP claims within 30 days.

q("""
SELECT
    a.member_id,
    a.claim_id                                                      AS first_admit,
    a.service_date                                                  AS first_date,
    b.claim_id                                                      AS readmit_claim,
    b.service_date                                                  AS readmit_date,
    CAST(JULIANDAY(b.service_date)-JULIANDAY(a.service_date) AS INT) AS days_between
FROM claims a
JOIN claims b
  ON  a.member_id = b.member_id AND a.claim_id != b.claim_id
  AND a.claim_type='IP' AND b.claim_type='IP'
  AND b.service_date > a.service_date
  AND JULIANDAY(b.service_date)-JULIANDAY(a.service_date) <= 30
WHERE a.claim_status='PAID' AND b.claim_status='PAID'
ORDER BY a.member_id
""", 'Q7 — 30-Day Readmission Proxy')

---
## Section 4: Data Quality

Data quality queries run as part of pipeline monitoring. Results feed into the QA log and alert stakeholders to issues that require remediation before data is used in reporting.

In [ ]:
# Business question: Are there data integrity issues in the claims?
# Why it matters: Denied claims with a non-zero paid amount indicate
# an unreconciled post-payment reversal — a billing system issue
# that overstates actual spend if not corrected.

q("""
SELECT
    claim_id, member_id, service_date,
    billed_amount, paid_amount, claim_status,
    'denied but paid_amount > 0' AS anomaly_flag
FROM claims
WHERE claim_status = 'DENIED' AND paid_amount > 0
""", 'Q8 — Data Quality: Denied Claims with Payment')

---
## Summary

These queries represent a standard analytics layer for a Medicaid managed care program:

| Query | Business Use |
|-------|-------------|
| Q1 — KPIs | Executive reporting, monthly dashboards |
| Q2 — Trend | Finance, budget forecasting |
| Q3 — PMPM | Plan performance monitoring |
| Q4 — Denial rate | Provider relations, billing audits |
| Q5 — BH utilization | Federal parity compliance reporting |
| Q6 — High-cost members | Care management targeting |
| Q7 — Readmission | Quality improvement, value-based payments |
| Q8 — Anomalies | Data quality monitoring, audit |

In a production environment these would run as **scheduled Databricks SQL jobs**, with results loaded into **Amazon QuickSight or Tableau** for stakeholder dashboards.